
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 강의 - MLflow의 평가 프레임워크

## 개요

MLflow는 생성형 AI 애플리케이션을 위해 특별히 설계된 포괄적인 평가 프레임워크를 제공하며, 자동화된 평가자(judges), 트레이싱(tracing) 기능, 체계적인 평가 도구를 지원합니다. 본 강의에서는 MLflow의 아키텍처와 핵심

MLflow 평가의 세 가지 기본 구성 요소를 살펴보고, `mlflow.genai.evaluate()` 함수의 개념을 파악하며, 트레이싱이 어떻게 정교한 평가 기능을 위한 기반을 제공하는지 탐색합니다.

**학습 목표**

본 강의를 마치면 여러분은 다음을 수행할 수 있습니다.
- MLflow 평가 프레임워크의 아키텍처와 주요 구성 요소를 설명할 수 있습니다.
- 평가 데이터셋, 스코어러(scorers), 예측(predict) 함수의 역할을 이해할 수 있습니다.
- `mlflow.genai.evaluate()` 함수가 평가를 어떻게 조율하는지 설명할 수 있습니다.
- 에이전트 평가 및 디버깅에서 트레이싱이 어떤 역할을 하는지 이해할 수 있습니다.
- AI 게이트웨이(Gateway) 연동을 통해 프로덕션 모니터링이 어떻게 이루어지는지 파악할 수 있습니다.

## A. MLflow 개요 및 OpenTelemetry 연동

### A1. MLflow 플랫폼 개요

![preparing-for-evaluation.png](../Includes/images/Evaluation with MLflow/preparing-for-evaluation.png "preparing-for-evaluation.png")

MLflow는 머신러닝 라이프사이클 전반을 관리하는 오픈소스 플랫폼입니다. 현재는 고도화된 관측 가능성(observability)과 모니터링을 위해 OpenTelemetry를 네이티브로 지원하며, 이를 통해 외부 모니터링 플랫폼으로 트레이스(trace)와 메트릭을 원활하게 내보낼 수 있습니다.

**MLflow 핵심 기능:**
- 매개변수(parameter), 메트릭, 모델 이력(lineage)을 포함한 실험 추적
- 버전 관리, 단계 전환, 주석 처리를 위한 모델 레지스트리(Unity Catalog와 연동 가능)
- 배치, 스트리밍, 실시간 추론을 위한 모델 배포
- 즉각적인 관측을 위한 실시간 추적 서버(MLflow Tracing)
- 생성형 AI(GenAI) 트레이스의 자동 스코어링을 포함한 운영 환경 모니터링
- 프롬프트 엔지니어링 및 생성형 AI 평가 워크플로우에 대한 심층적 지원

### A2. OpenTelemetry 연동

**OpenTelemetry 연동의 장점:**
- OpenTelemetry는 텔레메트리 데이터 수집 및 내보내기를 위한 개방형 표준으로, 클라우드 네이티브 시스템 전반의 모니터링을 위해 널리 채택되어 있습니다.
- MLflow 트레이스는 OpenTelemetry 트레이스 규격과 완전히 호환되므로, 널리 쓰이는 솔루션(예: Datadog, New Relic, Grafana, Splunk)으로 데이터를 내보낼 수 있습니다.
- MLflow는 세 가지 트레이스 내보내기 모드를 지원합니다:
  - MLflow 추적 전용(기본값): 트레이스를 MLflow 추적 서버로 전송
  - OpenTelemetry 전용: 트레이스를 OpenTelemetry Collector로 전송
  - 이중 내보내기: MLflow와 OpenTelemetry Collector 모두에 트레이스 전송

## B. 핵심 아키텍처

### B1. 세 가지 기본 구성 요소

<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://docs.databricks.com/aws/en/assets/images/flowchart-00c729ac75207b58d9c2243583a30d5a.png" alt="MLflow 평가">
</div>

MLflow는 생성형 AI 애플리케이션에 특화되어 설계된 종합적인 평가 프레임워크를 제공합니다. 이 아키텍처는 세 가지 핵심 구성 요소를 중심으로 작동합니다.

### B2. 구성 요소 1: 평가 데이터셋 (Evaluation Datasets)

평가 데이터셋은 테스트할 대상을 정의합니다. 최소한의 조건으로 입력값(에이전트에게 보내는 질의 또는 요청)을 포함해야 하며, 필요에 따라 다음과 같은 항목을 추가할 수 있습니다.

- **출력값(Outputs)**: 추론을 다시 실행하지 않고도 빠르게 평가할 수 있도록 미리 생성해 둔 에이전트의 답변
- **기대값(Expectations)**: 예상되는 사실관계, 예상 답변, 행(row)별 가이드라인과 같은 정답(Ground Truth) 정보
- **트레이스(Traces)**: 다단계 에이전트의 행동을 분석하기 위한 전체 실행 추적 기록
- **메타데이터(Metadata)**: 사용자 선호도, 대화 기록, 검색된 문서 등과 같은 추가적인 컨텍스트

데이터셋은 쉽게 수정하고 버전 관리를 할 수 있도록 주로 JSON 파일이나 Pandas DataFrame 형태로 저장됩니다.

### B3. 구성 요소 2: 스코어러 / 평가자 (Scorers / Judges)

스코어러는 정해진 기준에 따라 에이전트의 출력값을 평가합니다. MLflow는 다음과 같은 다양한 유형의 스코어러를 제공합니다.

- **내장형 평가자(Built-in judges)**: 정확성, 연관성, 안전성 등 일반적인 기준에 맞춰 연구 검증을 거친 LLM 기반의 평가 방식
- **가이드라인 평가자(Guideline judges)**: 자연어로 표현된 맞춤형 비즈니스 규칙
- **코드 기반 스코어러(Code-based scorers)**: 글자 수 확인, 형식 검증 등 결정론적 평가를 수행하는 Python 함수
- **사용자 정의 LLM 평가자(Custom LLM judges)**: 특수한 요구사항에 맞게 직접 설계한 LLM 기반의 평가 로직

### B4. 구성 요소 3: 예측 함수(Predict Function)

예측 함수는 평가 데이터셋에 대한 출력값을 생성합니다. 이는 다음과 같은 형태로 구성될 수 있습니다.

- 에이전트의 예측 메서드(실시간 평가용)
- 입력값을 에이전트가 요구하는 형식으로 변환하는 람다(lambda) 함수
- 미리 생성된 출력값을 평가하는 경우 완전히 생략 가능

이 세 가지 구성 요소가 `mlflow.genai.evaluate()` 내에서 하나로 결합하여 평가 프로세스를 조율하고, 메트릭을 수집하며, 분석을 위한 종합적인 결과를 기록합니다.

## C. `mlflow.genai.evaluate()` 함수

### C1. 중앙 오케스트레이션 포인트

```python
import mlflow
from mlflow.genai.scorers import Correctness

results = mlflow.genai.evaluate(
    data=eval_dataset,                  # 데이터프레임, 리스트[dict], 또는 평가데이터셋
    scorers=[Correctness()],            # 내장 및/또는 맞춤형 평가자
    predict_fn=my_app,                  # 선택 사항: 직접 평가
    # model_id="models:/my-app/1",      # 선택 사항: 버전 관리된 앱 연결
)
```

`mlflow.genai.evaluate()` 함수는 에이전트 평가를 위한 핵심 오케스트레이션(조율) 지점 역할을 합니다. 효과적인 평가 워크플로우를 구축하기 위해서는 이 함수의 동작 방식을 이해하는 것이 매우 중요합니다.

### C2. 주요 매개변수 (Key Parameters)

**주요 파라미터:**
- `data`: 평가 데이터셋
- `scorers`: 스코어링(점수 측정) 함수 목록
- `predict_fn`: 에이전트 또는 애플리케이션 함수
- `model_id`: 선택 사항인 모델 참조 ID

### C3. 평가 워크플로우(Evaluation Workflow)

**평가 워크플로우:**

1. **데이터 로드(Data loading)**: MLflow는 평가 데이터셋을 불러오고 그 구조를 검증합니다
2. **출력 생성(Output generation)**: `predict_fn`이 있으면 MLflow가 각 입력에 대해 이를 호출해 출력을 생성합니다
3. **트레이스 생성(Trace creation)**: `predict_fn`에 트레이스 기능이 구현되어 있거나(예: `@mlflow.trace` 또는 `mlflow.openai.autolog` 사용), `mlflow.genai.to_predict_fn` 을 통해 엔드포인트를 평가할 때 각 예측마다 MLflow 트레이스가 생성됩니다. '답안지(answer sheet)' 모드의 경우, 애플리케이션을 직접 실행하지 않더라도 입력과 출력을 바탕으로 트레이스를 구성합니다.
4. **스코어러 실행(Scorer execution)**: 각 스코어러가 자체 로직에 따라 입력, 출력, 트레이스를 평가합니다.
5. **결과 집계(Result aggregation)**: 개별 점수들이 요약 메트릭으로 집계됩니다.
6. **로그 기록(Logging)**: 분석 및 비교를 위해 결과가 MLflow에 기록됩니다.

### C4. 반환 값 및 결과 접근 (Return Value and Results Access)

**반환 값:**

함수는 다음 내용을 포함하는 `EvaluationResult` 객체를 반환합니다:
- **run_id**: 이 평가 실행에 대한 고유 식별자
- **metrics**: 모든 예시에 대해 집계된 메트릭(예: 평균 점수, 합격률)

**예시별 결과 접근:**

MLflow 3 버전에서는 result_df 속성 대신 트레이스를 통해 예시별 결과에 접근합니다.
```python
eval_traces = mlflow.search_traces(run_id=results.run_id)
```

이러한 구조화된 접근 방식을 통해 개별 에이전트 상호작용에 대한 완전한 가시성을 유지하면서도 체계적인 평가를 수행할 수 있습니다.

## D. MLflow Tracing: 에이전트 관측 가능성(Observability)의 기반

### D1. 종합적인 관측 가능성

MLflow Tracing은 에이전트 실행 과정에 대한 종합적인 관측 가능성을 제공하여, 에이전트의 추론 프로세스 내 모든 단계를 포착합니다. 트레이싱은 올바르게 인스투르멘테이션(계측)된 predict 함수와 함께 `mlflow.genai.evaluate()`를 사용할 때 활성화되며, 다양한 평가 기능의 근간을 이룹니다.

**추적이 포착하는 것들:**

- **모델 호출**: 프롬프트, 응답, 모델 파라미터를 포함한 파운데이션 모델과의 모든 상호작용
- **도구 호출**: 입력 파라미터와 반환값을 담은 함수 호출
- **정보 검색(Retrieval) 작업**: 콘텐츠 및 메타데이터를 포함하여 AI search 인덱스에서 검색된 문서
- **시간 정보**: 성능 분석을 위한 각 작업 지속 시간
- **계층 구조**: 실행 흐름을 보여주는 상위-하위(Parent-child) 관계

### D2. 트레이스 내 스팬(Span) 유형

MLflow는 트레이스를 다음과 같은 특정 유형의 스팬으로 구성합니다.
- **루트 스팬(Root span)**: 전체 에이전트 호출을 나타내는 최상위 스팬
- **ETRIEVER(리트리버)**: AI search또는 기타 검색 시스템에서 문서를 가져오는 스팬
- **TOOL(도구)**: 개별 도구 또는 함수 호출
- **CHAT_MODEL(챗 모델)**: 거대언어모델(LLM)과의 상호작용
- **CHAIN(체인)**: 일련의 작업 시퀀스 (LangChain 기반 에이전트에서 흔히 사용됨)

### D3. 평가에서 트레이싱이 중요한 이유

**왜 평가에서 트레이싱이 중요한가:**

`RetrievalSufficiency`(검색 충분성)와 같은 특정 평가 기준(Judge)은 정상적인 작동을 위해 트레이스를 필요로 합니다. 이러한 기준은 검색 시스템이 적절한 정보를 제공했는지 평가하기 위해 (최종 응답뿐만 아니라) 검색된 내용 자체를 분석합니다. 트레이스가 없다면 이러한 정교한 평가는 불가능합니다.

또한 트레이싱은 평가가 실패했을 때 정확히 어떤 일이 일어났는지 면밀히 조사할 수 있게 해줌으로써 디버깅을 가능하게 하며, 문제가 검색 품질, 도구 선택, LLM 추론 중 어디에서 기인했는지 식별할 수 있도록 돕습니다.

## E. AI Gateway 연동 및 운영 모니터링

### E1. AI Gateway 연동

Mosaic AI Agent Framework를 사용하여 에이전트를 등록하고 이를 Model Serving에 배포하면, Databricks가 AI Gateway 기능이 강화된 추론 테이블(inference tables)을 자동으로 활성화합니다. 이 테이블은 운영 환경에서 발생하는 모든 요청과 응답에 대한 상세한 로그를 제공합니다.

**추론 테이블의 장점:**

- **자동 로깅**: 별도의 추가 코드 구현 없이도 배포된 에이전트로 들어오는 모든 요청이 기록됩니다.
- **풍부한 메타데이터**: 요청 및 응답 내용, 타임스탬프, 지연 시간(latency), 모델 버전, 트레이스(trace) 데이터가 모두 포함됩니다.
- **쿼리 인터페이스**: Unity Catalog 내에서 SQL 쿼리를 통해 데이터를 조회할 수 있어 분석과 모니터링이 용이합니다.
- **평가 통합**: 추론 테이블의 데이터를 평가용 데이터셋으로 바로 활용할 수 있습니다.

### E2. 운영 데이터의 평가 활용

이러한 연동을 통해 강력한 피드백 루프가 구축됩니다.
1. 에이전트는 프로덕션에서 실행되며 모든 상호작용을 추론 테이블에 기록합니다
2. 추론 테이블을 쿼리하여 흥미로운 예시, 실패 사례, 또는 예외 사례를 추출합니다
3. 이러한 실제 사례가 평가 데이터셋을 보강합니다.
4. 향후 진행되는 평가는 이처럼 실제 운영 시나리오를 반영하게 됩니다.

이러한 방식을 통해 평가 데이터셋이 정체되거나 뒤처지지 않고, 실제 사용자 행동에 맞춰 지속적으로 발전할 수 있습니다.

## F. 핵심 요약

MLflow의 평가 프레임워크는 다음과 같은 요소를 통해 에이전트 평가를 위한 종합적인 솔루션을 제공합니다.

1. **3단계 아키텍처**: 평가 데이터셋, 스코어러(Scorer), 예측 함수가 서로 유기적으로 연동됩니다.
2. **중앙 집중식 오케스트레이션**: `mlflow.genai.evaluate()`를 통해 복잡한 평가 워크플로우를 손쉽게 처리합니다.
3. **종합적인 추적(Tracing)**: 에이전트 실행 과정을 전체적으로 시각화하여 정밀한 평가와 디버깅이 가능해집니다.
4. **운영 환경 통합**: AI 게이트웨이 및 추론 테이블을 활용해 운영 환경과 평가 단계 간의 피드백 루프를 구축합니다.
5. **OpenTelemetry 호환성**: 업계 표준 관측 가능성(Observability) 도구와 원활하게 연동됩니다.

이러한 기반 기술을 바탕으로 에이전트 개발 요구사항에 맞춰 확장 가능한 체계적인 평가를 수행할 수 있습니다. 다음 강의에서는 이 프레임워크 내에서 구현할 수 있는 구체적인 평가 모델(Judge)의 종류와 평가 전략에 대해 알아보겠습니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>